<a href="https://colab.research.google.com/github/DavidSenseman/BIO1173/blob/main/BIO1173_Class_05_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<!-- BIO1173_Class_05_3:Rev 1 -->

---------------------------
**COPYRIGHT NOTICE:** This Jupyterlab Notebook is a Derivative work of [Jeff Heaton](https://github.com/jeffheaton) licensed under the Apache License, Version 2.0 (the "License"); You may not use this file except in compliance with the License. You may obtain a copy of the License at

> [http://www.apache.org/licenses/LICENSE-2.0](http://www.apache.org/licenses/LICENSE-2.0)

Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.

------------------------

# **BIO 1173: Intro Computational Biology**

## **Module 5 : Image Generation and Analysis**

* Instructor: [David Senseman](mailto:David.Senseman@utsa.edu), [Department of Biology, Health and the Environment](https://sciences.utsa.edu/bhe/), [UT San Antonio](https://www.utsa.edu/)

### Module Content

* Class_05_1 : Cloud-based Models
* Class_05_2 : Image Generation
* **Class_05_3 : Image Analysis**
* Class_05_4 : Video Generation

## Google Colab Instructions

Run next code cell to map this Colab lesson's folder /content/drive to your Google Drive. This will allow you keep a copy of this Colab notebook which is your course textbook.

In [ ]:
# @title **Run this cell first**
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    Colab = True
    print("Note: Using Google CoLab")
    !curl ipinfo.io
except:
    print("**WARNING**: Your GDrive is not mapped to /content/drive ")
    print("**WARNING**: Your work will not be saved!")
    Colab = False

You should see something _similar_ to the following output:
```text
Mounted at /content/drive
Note: Using Google CoLab
{
  "ip": "34.139.115.187",
  "hostname": "187.115.139.34.bc.googleusercontent.com",
  "city": "North Charleston",
  "region": "South Carolina",
  "country": "US",
  "loc": "32.8546,-79.9748",
  "org": "AS396982 Google LLC",
  "postal": "29415",
  "timezone": "America/New_York",
  "readme": "https://ipinfo.io/missingauth"
}
```

## **Get Course Information**

Run the code in the cell below to retrieve your BIO1173 Course Key and your port number.

In [ ]:
# @title **Run this Cell to Get Your Course Information**
from google.colab import userdata
import urllib.request
import time
import os


myUTSA_ID = userdata.get('myUTSA_ID')
os.environ['myUTSA_ID'] = myUTSA_ID

URL = ("https://biologicslab.co/BIO1173/backend_code/send_student_info.py?v="
        + str(time.time()))
try:
  req = urllib.request.Request(
      URL, headers={"Cache-Control": "no-cache, no-store, must-revalidate"})
  with urllib.request.urlopen(req, timeout=20) as r:
    exec(r.read().decode("utf-8"))
except Exception as exc:
  print(f"\u26a0\ufe0f  Could not load the course-information cell: {exc}")
  print("    Check your internet connection and run this cell again.")


# **Image Analysis**



##**Historical Development AI Image Analysis**


####**Rule-based Algorithms**

The origins of artificial intelligence in image analysis date back to the 1960s, driven by early ambitions to mathematically replicate human vision. During this period, researchers relied on symbolic AI and **hand-crafted algorithms** to identify basic shapes, lines, and edges. This era is famously highlighted by MIT's 1966 **"Summer Vision Project,"** which optimistically aimed to solve foundational computer vision problems in just a few months. For decades, however, progress was heavily constrained by limited computational power and the rigid nature of rule-based systems, which struggled to process the vast complexity, lighting shifts, and spatial variability of real-world visual data.

####**Statistical Machine Learning**

The late 1990s and 2000s marked a crucial transition from strict rule-based systems to **statistical machine learning** approaches. Engineers began utilizing sophisticated mathematical techniques for manual feature extraction, such as the Scale-Invariant Feature Transform (SIFT) and the Histogram of Oriented Gradients (HOG), which were then paired with classifiers like Support Vector Machines (SVMs). This era yielded significant practical breakthroughs, most notably the **Viola-Jones object detection framework** in 2001, which brought reliable, real-time face detection to consumer digital cameras. Despite these successes, the field was still bottlenecked by human limitations, as systems required extensive domain expertise to manually engineer the specific visual features the algorithms needed to track.

####**Convolutional Neural Networks (CNNs)**

The landscape fundamentally shifted in 2012 when the deep learning revolution took hold, catalyzed by a Convolutional Neural Network (CNN) called **AlexNet** that decimated previous records in the ImageNet visual recognition competition. Instead of relying on human-engineered features, CNNs automatically learned to identify increasingly complex visual patterns by processing massive, labeled datasets using graphics processing units (GPUs). This breakthrough catalyzed an explosion of modern image analysis capabilities, rapidly evolving into specialized architectures for real-time object detection and segmentation, and eventually culminating in today's sophisticated Vision Transformers (ViTs) and multimodal foundation models capable of complex semantic reasoning across both visual and textual inputs.

## **`WeVisOne-VL-4B` Vision Language Model**

This lesson is designed to show you how you can analyze images with a modern vision language (VL) model, using the recently released **WeVisOne-VL-4B** attached to your Hermes Agent.

`WeVisOne-VL` is a powerful VLM that unifies object-level and pixel-level perception through structured next-token prediction. It supports multiple visual tasks including object detection, referring localization, counting,
OCR, segmentation, and matting, enabling localization at any user-specific granularity, from points and boxes to masks and alpha mattes.

###**Data Engine**

The model uses a unified data engine that connects semantic discovery with instance-aligned supervision for object detection, referring localization, segmentation, and matting. To guide targeted acquisition and task-specific annotation, the engine organizes category concepts and compositional referring expressions in a shared knowledge graph.

![__](https://biologicslab.co/BIO1173/images/class_05/WeVisOne_Paper_Fig2.png)

The model then combines complementary localization experts to generate spatial candidates and consolidates their predictions through visual-language verification of query–instance correspondence. Building on these verified localizations, the engine generates binary masks to extend supervision to the pixel level while preserving target identity across tasks, with manual spot checks for additional qualitycontrol. For matting, it further leverages expert predictions and curated data to support a dedicated annotator that provides continuous alpha supervision. Finally, the engine uses reviewed model errors to guide subsequent acquisition, annotation correction, and hard-example sampling, enabling iterative refinement of the training data.


###**Model Architecture**

Given an image and an instruction, WeVisOne-VL first encodes the image with the Qwen3-VL vision encoder and autoregressively generates the structured response described above. For pixel-level tasks, it invokes a lightweight decoder for each generated <OBJ> token. The decoder takes the token’s hidden state, its predictedbox, and multi-level features from the Qwen3-VL vision encoder.

![__](https://biologicslab.co/BIO1173/images/class_05/WeVisOne_Paper_Fig4.png)

The decoder draws image context from the shared visual features and spatial guidance from the predicted boxes, eliminating the need for a separate large segmentation model such as SAM



##**Log Into Your LXC Container**

Run the next cell to generate your specific ssh command for this lesson.

In [ ]:
# @title Generate ssh command
print(f"ssh -o ExitOnForwardFailure=yes -L 8765:localhost:8765 -p {PORT_NUM} {myUTSA_ID}@lucy.utsa.edu")

Copy the **`ssh command`** above and paste it into your terminal window to log into your LXC container.

## **Start Hermes Agent**

Start your Hermes **Terminal User Interface (TUI)** by entering this command in your terminal:
```text
hermes --tui
```
Use the following command to give your session the appropiate title.

```text
/new BIO1173_Class_05_3
```

Say **Yes** when you asked to start a new session:
```text
⚠ Start a new session?
 This ends the current conversation and clears the transcript.
  No, keep going
▸ Yes, start a new session
↑/↓ select · Enter confirm · Y/N quick · Esc cancel
  
 ```
You should now see **BIO1173_Class_05_3** at the bottom right of your TUI.

##**Image Analysis with `WeVisOne`**

When performing image analysis, `WeVisOne` can take advantage of five aspects of referring object detection: spatial position, world knowledge, fine-grained attributes, negative attributes, and absent-target queries.

###**The Reference Image**

We will use this reference image to demonstrate some of the capabilities of the `WeVisOne-VL-4B`. This is an image of a pride of lions consisting of 3 lionesses (adult females) and 4 dependent cubs (juveniles). There are no adult male lions in this image.

![__](https://biologicslab.co/BIO1173/images/class_05/LionFamily.jpg)

###**Quantitative Analysis**

We start with quantitative analysis, or **_counting_** objects. This requires your Hermes Agent to first recognize the specified object, and then count the number of instances that the object occurs in the image. In this example, we take advantage of `WeVisOne` world knowledge, to correctly identify a collection of pixels as a "lion". To make this analysis more demanding, we will ask Hermes only to count the number of lionesses. This will require Hermes to differentiate between adult and juvenile lions (cubs) as well as to correctly identify the sexual dimorphism displayed by adult males and females (lionesses).

Start by giving your Hermes Agent this prompt.

**Hermes Prompt:**
```text
Use your terminal tool to run: curl -sSL -o ~/LionFamily. jpg
http://biologicslab.co/BIO1173/images/class_05/LionFamily.jpg
Then use the locate tool on ~/LionFamily.jpg with target "each
adult female lion, whole body, not the cubs". Count the boxes it
returns and tell me: how many adult female lions are in this
image? Then give me the view_url of the annotated image so I can
open it in my browser.

```
If you are using a Windows PC use **`Ctrl+Shift+V`** to paste text into your Hermes Agent TUI. If you are using a MacBook, use **`Cmd+V`** to paste the text.

Press **`Enter/Return`** to send it to your Hermes Agent.

---------------

####**Viewing Analyzed Images**

>To see the analyzed image, open a new browser tab on your laptop and point it at: http://localhost:8765/ to see the list of images. You may have to refresh the page after each new image is generated.

-------------

Here is image analysis generated by Hermes Agent for the first prompt:

![__](https://biologicslab.co/BIO1173/images/class_05/class_05_3_image01A.png)

Hermes correctly placed a red bounding box around each of the 3 lionesses.

-------------

###**Object Pointing**

This next example, we will illustrate the model's ability to peform relational, attribute-based, and fine-grained pointing queries. In other words, Hermes not only knows which pixels constitute a particular object (e.g. a lion cub) but whether the lion cub is standing or sitting, and whether the lion cub is, or is not, looking directly at the camera.

Send this prompt to your Hermes Agent.

**Hermes Prompt:**
```text
Reuse the last prompt to find the lion club sitting on the ground directly facing the camera.
```

Here is the processed image.
![__](https://biologicslab.co/BIO1173/images/class_05/class_05_3_image02A.png)

Hermes was able to accurately locate **`1`** lion cub sitting on the ground and directly facing the camera.

------------

####**Reporting Null Results**

A serious problem with all LLMs is their tendency to **make up results**. Even the best frontier models will sometimes fabricate results -- we say that the model is **"halucinating"**.

With Vision-Language (VL) models, halucinations are more like to occur when the model is asked to find something that does not appear in the model. A good model will say that something does not occur in an image whereas a poor model will incorrectly report the object was found.

Let's see what happens if you ask Hermes Agent to count the number of adult male lions in the reference image using the next prompt.

**Hermes Prompt:**
```text
Reuse the prompt but change it to count the number of adult male lions.
```

Here is the response from Hermes Agent.

**TUI Output:**
```text
0 adult male lions were found in the image.                   
                                                              
The detection returned no matches (count = 0, found: false).  
    
```
Hermes did **not** halucinate. It correctly reported **`0`** adult male lions in the image.


##**The One Health Initiative (OHI)**

In this section we look at some of the specific **`use cases`** of image analysis within the context of the BHE Department's **One Health Initiative (OHI)**.

####**Management of Wildlife Populations**

One OHI focus concerns the management of wildlife populations. An obvious metric that needs to established is population size.

In this example we ask Hermes Agent to count the number of birds in a flock.

Send this prompt to your Hermes Agent.

**Hermes Prompt:**
```text
Use your terminal tool to run: curl -sSLf -o ~/BirdFlock.png
http://biologicslab.co/BIO1173/images/class_05/BirdFlock.png &&
file ~/BirdFlock.png Then use the locate tool on ~/BirdFlock.png with target
"all birds". Count the boxes or points it returns and tell me how many
birds there are in this image. Then give me the view_url of the annotated image.
```
Here is the result from one run.

**TUI Output:**
```text
Birds found: 99
```

Here is the corresponding image showing the locations of the inidividual birds identified by the model.
![__](https://biologicslab.co/BIO1173/images/class_05/class_05_3_image03A.png)

This is a good result but with clear limits. Hermes reported 99 correctly: the tool did return 99 boxes. You can treat 99 as a decent estimate and not as an exact count.

####What Hermes did well

* Isolated birds are almost all boxed, tightly. This covers the scattered ones across the lower half, the lone bird at the top, and most of the left side.

* The model mostly separates individual birds even in the flock, which the point version at the smaller size struggled with.

####Where Hermes falls short

* Missed birds in the crowded band. At least five are visibly unboxed, for example the elongated birds near the center and upper-right of the cluster, and the one cut off at the far left edge. The thick chains across the top and right are the worst-covered area.

* Duplicate and overlapping boxes. Several birds have two boxes stacked on them (around the upper-middle and the right-hand side), and some boxes contain two overlapping birds.

* Boxes that clip the wings. On diagonal birds the box often catches the body but not the wingtips.    

Overall, misses and duplicates push the count in opposite directions, so 99 may land near the truth by luck. A manual count would probably be somewhat higher. It’s one run, so I wouldn’t read more precision into it





##**Additional Use Cases for Image Analysis**

Because **One Health** focuses on the intersection of human, animal, and environmental health, computer vision can be deployed across all three domains simultaneously to predict and track spillover events for pathogens relevant to South Texas, such as West Nile Virus, Avian Influenza, or Chagas disease.

Here is how image analysis could be applied across the One Health triad:

**1. Environmental Health: Geospatial Vector Mapping**
*   **The Application:** Processing automated satellite and drone imagery of Bexar County and surrounding regions to identify ecological risk factors.
*   **The Mechanism:** Vision models can be trained to detect standing water accumulation, changes in vegetation indices, and zones of rapid urban encroachment into wildlife habitats. This allows researchers to predict where mosquito or kissing bug (triatomine) breeding grounds will overlap with human populations.

**2. Animal Health: Behavioral and Phenotypic Monitoring**
*   **The Application:** Using camera traps or agricultural video feeds to monitor local wildlife, livestock, or regional poultry flocks.
*   **The Mechanism:** Image analysis models can flag early physical or behavioral indicators of disease—such as lethargy, abnormal gait, or isolation from a flock—before widespread transmission occurs. This acts as an early warning system for pathogens like Avian Flu before they can cross over into human populations.

**3. Human Health: High-Throughput Microscopic Pathology**
*   **The Application:** Automated screening of digitized blood smears or tissue samples in BHE laboratories.
*   **The Mechanism:** Localized AI inference engines can rapidly scan high-resolution microscopic images to detect the presence of zoonotic parasites or cellular abnormalities, drastically reducing the time required for manual slide review.



##**Optical Character Recognition (OCR)**

**Optical Character Recognition (OCR)** is a computer vision technology that converts images of typed, handwritten, or printed text into machine-encoded, searchable, and editable data. By analyzing the shapes and patterns of characters in scanned documents, PDFs, or photographs, OCR bridges the gap between physical records and digital computation, allowing algorithms to process and analyze text that was previously locked in static images.

We will be looking more closely at OCR later in the course. Here we just want to illustrate an important use case of converting a hand-written note to text. For this particular example we will be using this nurse's note:

![__](https://biologicslab.co/BIO1173/images/class_05/nurses_note.png)

Send this prompt to your Hermes Agent to convert nurse's note into text.

**Hermes Prompt:**
```text
Use your terminal tool to run: curl -sSLf -o ~/nurses_note.png http://
biologicslab.co/BIO1173/images/class_04/nurses_note.png && file ~/nurses_note.
png
Then use your terminal tool to run exactly this command:
python3 -c "import base64,json,re,os,urllib.request as u;d=base64.b64encode
(open(os.path.expanduser('~/nurses_note.png'),'rb').read()).decode();b
{'model':'wevisone-vl-4b','temperature':0,'max_tokens':4000,'messages':
[{'role':'user','content':[{'type':'text','text':'Read all the text in this
image.'},{'type':'image_url','image_url':{'url':'data:image/png;base64,'
+d}}]}]};r=u.Request('http://192.168.1.20:8000/v1/chat/completions',json.dumps
(b).encode(),{'Content-Type':'application/json'});t=json.loads(u.urlopen(r,
timeout=300).read())['choices'][0]['message']['content'];m=sorted([(a,(int(y1)
+int(y2))/2,int(x1)) for a,x1,y1,x2,y2 in re.findall(r'<ref>(.*?)</ref><box><
(\d+)><(\d+)><(\d+)><(\d+)></box>',t)],key=lambda z:z[1]);w=[];[w[-1].append\
(i) if w and i[1]-w[-1][0][1]<22 else w.append([i]) for i in m];print('\n'.
join('  |  '.join(z[0] for z in sorted(g,key=lambda z:z[2])) for g in w))"
Show me the command's output exactly as printed, line by line. Do not
correct, summarize or interpret it.
```

Here is the output generated by the test student's Hermes Agent.

**TUI Output:**
```text
  Pronunciated August 1948  |  Standard Form 510                          
  By Bureau of the Budget  |  Circular A-32                               
  CLINICAL RECORD  |  NURSE'S NOTES  |  (Sign all notes)                  
  AND HOUR  |  DATE  |  MEDICATION-TREATMENT  |  (When indicated include
  intake and output, type, amount, and time)  |  
  OBSERVATIONS                  
  4-96-48  |  wand just recieved and tosy                                 
  a bite out of it. Then                                                  
  we ask for and now please                                               
  os taste. Pt. drank 250 cc                                              
  of orange juice in forty-                                               
  Dr.Hip. flower                                                          
  malvin, standing by (WB).                                               
  1130  |  Pt. up to head - Jusuing e                                     
  2 lowering.                                                             
  1200  |  TPR 98° - 76 - 20  |  Pt. resting in bed - remarked            
  about card he had taken a                                               
  bite out of.                                                            
  1220  |  Relieved forshow.                                              
  Lunch.  |  sawed - pt. qte good.                                        
  1305  |  Returned from chow.                                            
  1400  |  Pt. up in chair reading book.                                  
  Tumunited Hol 100                                                       
  1500  |  Lunon av. 14  |  given as ordered. Pt. up                      

```


##**Generate Session Summary**


####**Step 1: Generate Session Summary**
In order to grade this Colab lesson, give this final command to your Hermes Agent to generate a summary of your work:

```text
/generate-session-summary
```
----------

####**Step 2: Copy Session Summary to your Clipboard**

Copy the output generated by your Hermes Agent into your clipboard. When you are selecting text from the TUI, remember, if you are using a Windows laptop, to hold down three keys: **`Shift+Ctrl+Alt`** while you use your mouse/trackpad to select the text. If you have a MacBook, you just need to hold down these two keys: **`fn+option`** while you select your text. The selected text should turn white as shown in this image:

![__](https://biologicslab.co/BIO1173/images/class_01/select_session_summary.png)


Then press **`Shift+Ctrl+C`** to copy the selected text to your clipboard if you are using a Windows laptop. If you are using a MacBook, you just need to use **`Cmd+C`** to copy the text to your clipboard.

-------------

####**Step 3: Paste Session Summary into Generate Summary Code Cell**

Position your cursor/pointer immediately below this line in code cell below:
```text
# Paste your Session Summary below the next line
session_summary = r"""
```
Press **`Ctrl+V`** or **`Cmd+V`** to paste your summary.

**NOTE: This is a large session summary. You will likely need to copy-and-paste different sections of the output. Be careful when you "splice" each section together, that you don't lose any text or repeat any text.**

In [ ]:
# @title Generate Session Summary

# Paste your Session Summary below the next line
session_summary = r"""



"""

# Display Session Summary
print("Session Summary:")
print(session_summary)

# **Electronic Submission**

When you run the code in the cell below, it will grade your Colab notebook and tell you your pending grade as it currently stands. You will be given the choice to either submit your notebook for final grading or the option to continue your work on this class lesson.

Grant Access to your Colab Secrets if you are asked to do so.

In [ ]:
# @title  Electronic Submission

import urllib.request
import ssl
import time

url = "https://biologicslab.co/BIO1173/validate_session.py?v=" + str(time.time())
ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE
req = urllib.request.Request(
    url,
    headers={
        "Cache-Control": "no-cache, no-store, must-revalidate",
        "Pragma": "no-cache",
        "Expires": "0"
    }
)
with urllib.request.urlopen(req, context=ctx) as r:
    exec(r.read().decode("utf-8"))
main()